In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from boardgames_recsys.data.filtering import filter_df
from boardgames_recsys.data.matrix import *

from surprise import NMF
from surprise import Dataset
from surprise.reader import Reader
from sklearn.cluster import KMeans
import seaborn as sns
from sklearn.manifold import TSNE

from PIL import ImageColor
import plotly.express as px

sns.set_theme()

In [ ]:
folder = "../database_cleaned"
jeux_clean  = pd.read_csv(f"{folder}/jeux_clean.csv", index_col=0)
avis_clean  = pd.read_csv(f"{folder}/avis_clean.csv", index_col=0)
users       = pd.read_csv(f"{folder}/users.csv", index_col=0)

min_reviews = 10 # change to set one
# filter data with the minimum reviews
rev_filter = filter_df(avis_clean, min_reviews)

## NNMF -> 20 latent factors

In [ ]:
model = NMF(n_factors=20, random_state=42, biased=False, reg_pu= 0.1, reg_qi= 0.1)
data = Dataset.load_from_df(rev_filter[["User id", "Game id", "Rating"]], reader=Reader(rating_scale=(0, 10)))
trainset = data.build_full_trainset()
nmf = model.fit(trainset)

# Extract matrices
U = nmf.pu  # User-feature matrix (W)
G = nmf.qi  # Item-feature matrix (H)

games_ids = np.array([trainset.to_raw_iid(i) for i in range(len(G))])
users_ids = np.array([trainset.to_raw_uid(u) for u in range(len(U))])
G = G[np.argsort(games_ids), :]
U = U[np.argsort(users_ids), :]

### 30 Clusters and t-SNE (perplexity = $40$)

In [ ]:
sns.set_theme(rc={"figure.figsize":(6, 5)})
NB_CLUSTERS = 30
kmeans = KMeans(n_clusters=NB_CLUSTERS, random_state=42) 
distances = kmeans.fit_transform(G) 
game_labels = kmeans.labels_ 
G_embedded = TSNE(n_components=3, perplexity=40, max_iter=2000, random_state=1).fit_transform(G)

### Pushing clusters away from each other

In [ ]:
centers_pos = {

    24 : [0, 0, 10],  # best  
    10 : [0, 0, -10], # worst 

    1  : [-20, -20, -20], # collecte 
    26 : [-20+4, -20+4, -20],
    17 : [-20+4, -20+4, -20+4],

    28 : [20, -20, -20],  # atmosphere, settings 
    7  : [16, -16, -20],
    6  : [18, -18, -16],

    12  : [-20, 20, -20], # civilisation
    18  : [-16, 16, -20], 
    21  : [-18, 18, -16], 

    5   : [-20, 20, 20], # guerre

    3   : [0, 20, 0], # capture territoire 
    11  : [0, 16, -2], 
    27  : [-2, 20, 2], 

    2   :[20-4, 20+4, 20], # complexe
    9   :[20-2, 20-2, 20-2], 
    13  :[20+4, 20-4, 20], 
    25  :[20+4, 20+4, 20-2], 
    29  :[20, 20, 20+4], 

    22  : [20, -20, 23], # logique, déduction
    23  : [20, -20, 20],

    4 : [0, -20, -2], # great visuals
    8 : [0, -20, 2], 

    14 : [-20, -2, -2], # rapide & tactique
    20 : [-20, 2, 2],
 
    15 : [20, -2, 0], # eurogames 
    16 : [20, 2, 0], 

    0 : [20, 20, -16], # construction & expansion
    19 : [20, 20, -20],
}

clusters_groups = [[0, 19], [22, 23], [3, 11, 27], [2, 9, 13, 25, 29], [14, 20], [15, 16], [4, 8], [12, 18, 21], [1, 8, 17, 26], [6, 7, 28]]

In [ ]:
def translate(centers_pos, centers, points, labels):
    for cluster, center in enumerate(centers):
        translate_vector = np.array(centers_pos[cluster]) * 10 - center

        mask = (labels == cluster)
        points[mask] += translate_vector
        centers[cluster] += translate_vector

def recalc_centers(clusters_groups, translated_centers):
    groups_centers = []
    for group in clusters_groups:
        groups_centers.append(translated_centers[group, :].mean(axis=0))
    return np.array(groups_centers)

def push_points(centers, centers_groups, points, labels):
    for group_idx, group in enumerate(clusters_groups):
        for cluster in group:
            cluster_mask = (labels == cluster)
            direction = centers[cluster] - centers_groups[group_idx]
            if np.linalg.norm(direction) == 0:
                continue
            unit_vector = direction / np.linalg.norm(direction)
            points[cluster_mask] += unit_vector * 20

centers = []
for cluster in np.sort(np.unique(kmeans.labels_)):
    mask = (kmeans.labels_ == cluster)
    centers.append(G_embedded[mask].mean(axis=0))
centers = np.array(centers)

In [ ]:
# Move centers
centers_copy = centers.copy()
translated_points = G_embedded.copy()

translate(centers_pos, centers_copy, translated_points, kmeans.labels_)

# Calc center for each cluster group
centers_groups = recalc_centers(clusters_groups, centers_copy)

# Push points away from each other
push_points(centers_copy, centers_groups, translated_points, kmeans.labels_)
np.save("tsne_pushed.npy", translated_points)

### Generate data for app (t-SNE)

In [ ]:
matrix_ratings, mask_ratings, users_table, games_table = get_matrix_user_game(rev_filter)
users_table

In [ ]:
clusters = kmeans.labels_
themes = {
    "Tout les clusters": [],
    "🧩🕵️ Logique et Déduction": [22, 23],
    "🏗️🏰 Construction & expansion": [0, 19],
    "🧠⚡ Rapide & Tactique": [14, 20],
    "📚⏳ Longs & complexes": [2, 9, 13, 25, 29],
    "💎🃏 Collecte":  [1, 17, 26],
    "🪖💣 Guerre": [5],
    "🏛️🎲 Eurogames": [15, 16],
    "🌍🏺 Civilisation": [12, 18, 21],
    "🗡️🚩 Capture territoire": [3, 11, 27],
    "🌅🖼️ Superbes visuels": [4, 8],
    "🐉📜 Culture | Fantaisie": [6, 7, 28],
    "👎 Coups de blues": [10],
    "🏆 Coups de coeurs": [24],
}
themes_inverse = {cluster: theme for theme, clusters in themes.items() for cluster in clusters}


colors = {22: "#6B4C9A", 23: "#B39DDB",  # logique, déductions
          0: "#99582a", 19: "#ffe6a7",  # construction expansion
          14: "#fb8500", 20: "#ffb703",  # rapide, tactique
          2: "#48cae4", 9: "#00b4d8", 13: "#0096c7", 25: "#023e8a", 29: "#0077b6",
          1: "#1e96fc", 17: "#fcf300", 26: "#a2d6f9",  # collecte
          5: "#4B4B4B",  # guerre
          12: "#8C6D31", 18: "#C2B280", 21: "#4C6B5C",  # civilisation
          15: "#4E79A7", 16: "#A6C8E0",  # eurogames
          3: "#E15759", 11: "#F28E8E", 27: "#B63A3A",  # capture territoire
          4: "#ffc300", 8: "#ffd60a",  # super visuals
          6: "#00509d", 7: "#38b000", 28: "#A77DC2",  # cultural, fantastic settings
          24: "#f00000",
          10: "#000000",
          }

colors_points = [list(ImageColor.getcolor(colors[cluster], "RGB")) for cluster in clusters]

In [ ]:
color_palette_30 = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b",
    "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#393b79", "#637939",
    "#8c6d31", "#843c39", "#7b4173", "#5254a3", "#6b6ecf", "#9c9ede",
    "#8ca252", "#b5cf6b", "#cedb9c", "#bd9e39", "#e7ba52", "#e7969c",
    "#ad494a", "#a55194", "#ce6dbd", "#de9ed6", "#6b6b6b", "#c7c7c7"
]
label_names = {str(i): f"Cluster {i}" for i in range(30)}

colors = [color_palette_30[i] for i in kmeans.labels_]
games_info = pd.DataFrame(data={"x":translated_points[:, 0], "y":translated_points[:, 1], "z":translated_points[:, 2],
                                "color":[color_palette_30[i] for i in kmeans.labels_],
                                "size":[1] * translated_points.shape[0], "cluster":kmeans.labels_})

df_all = pd.DataFrame(data={
    "game_id": games_table.values,

    "x": translated_points[:, 0].tolist(),
    "y": translated_points[:, 1].tolist(),
    "z": translated_points[:, 2].tolist(),

    "color": colors_points,
    "cluster": clusters,
    "name": [themes_inverse[cluster] for cluster in clusters],
    "game index":  games_table.index
})
df_all.to_json("games_info.json", orient="records", force_ascii=False)



### Games for app

In [ ]:
jeux_clean = jeux_clean[jeux_clean["Game id"].isin(rev_filter["Game id"])]
games = jeux_clean[["Game id", "Game name year", "Description", "Type", "Min number of players", "Max number of players", "Age min", "Age max"]]
np.sort(games["Max number of players"].unique())

In [ ]:
ages = games[["Age min", "Age max"]]
ages_text = []
for age_min, age_max in ages.itertuples(index=False):
    txt = "-"
    if (not np.isnan(age_max)) and (not np.isnan(age_min)):
        if age_max <= 25:
            txt = f"{int(age_min)}-{int(age_max)}"
        else:
            txt = f"{int(age_min)}+"
    
    if np.isnan(age_max) and (not np.isnan(age_min)):
        txt = f"{int(age_min)}+"
    
    ages_text.append(txt)

In [ ]:
players = games[["Min number of players", "Max number of players"]]
players_text = []
for pl_min, pl_max in players.itertuples(index=False):
    txt = "-"
    if (not np.isnan(pl_min)) and (not np.isnan(pl_max)):
        if pl_max <= 18:
            txt = f"{int(pl_min)}-{int(pl_max)}"
        else:
            txt = f"{int(pl_min)}+"
    
    if np.isnan(pl_max) and (not np.isnan(pl_min)):
        txt = f"{int(pl_min)}+"
    
    players_text.append(txt)
print(len(players_text))
players_text

In [ ]:
means = rev_filter.groupby("Game id")["Rating"].mean().reset_index()
games = games.assign(Players=players_text, Age=ages_text)
games = games.merge(means, on="Game id")
games = games.drop(columns=["Age min", "Age max", "Min number of players", "Max number of players"])
games.loc[:, "Type"] = games["Type"].fillna("-")

games.loc[:, "Type"] = games["Type"].str.split("|").apply(lambda row:", ".join(row[:2]))
games = games.sort_values(by="Game id")
games["Game index"] = list(range(games.shape[0]))
#games
games

## Users

In [ ]:
users = pd.read_csv(f"{folder}/users.csv", index_col=0)
true_ratings, mask_ratings, users_table, games_table = get_matrix_user_game(rev_filter)

users_table = users_table.reset_index().rename(columns={"index":"User index"})
print(users_table)
users_count = rev_filter.groupby("User id")["Game id"].count().reset_index().sort_values("Game id").rename(columns={"Game id":"Number reviews"})
special_id = 9701
nb_users = {0:9, 1:10, 2:10, 3:7}

slices_max = [50, 100, 200, 1700]
slices_min = [10, 50, 100, 200]

users_ids = [special_id, 208, 201, 1191]
users_count.tail(10)
np.random.seed(1)
for i in range(len(slices_max)):
    mx, mn = slices_max[i], slices_min[i]
    slice_users = users_count[(users_count["Number reviews"] < mx) & (users_count["Number reviews"] >= mn) & (~users_count["User id"].isin(users_count))]["User id"]
    users_ids += slice_users.sample(nb_users[i], replace=False).values.tolist()

users_indices = users_table[users_table["User id"].isin(users_ids)].index
users_indices

### Predicted ratings

In [ ]:
pred = U @ G.T - 2
pred = np.clip(pred, a_min=0, a_max=10) / (10 - 2)
pred = pred[users_indices]
np.save("nnmf_prediction.npy", pred)
pred

In [ ]:
users = users[users["User id"].isin(rev_filter["User id"])].sort_values("User id")
users = users.merge(users_table, on="User id").merge(users_count, on="User id")
users = users[users["User id"].isin(users_ids)]
users

In [ ]:
# ratings = pred[0, :]
# red, green, blue = (255 * (1 - ratings)).astype(int), (255 * ratings).astype(int), np.zeros(ratings.shape, dtype=int)

# games["color"] = [[r, g, b] for r, g, b in zip(red, green, blue)]
# green[green > 255]

In [ ]:
users_games = rev_filter[["User id", "Game id"]].merge(games[["Game id", "Game index"]], on="Game id")[["User id", "Game index"]]
users_games["Game index"] = users_games["Game index"].astype(int)
users_games = users_games.groupby("User id").agg(list).reset_index().rename(columns={"Game index":"Rated games index"})
users = users.merge(users_games, on="User id")
users

In [ ]:
pred = U @ G.T
#print(pred.shape)
users_top_games = []
users_top_ratings = []
for user in users_indices:
    
    indices = np.arange(0, pred.shape[1]) # all indices
    rated_games = users[users["User index"] == user]["Rated games index"].item() # already rated games
    not_rated_games = np.setdiff1d(indices, rated_games)
    
    ratings = pred[user, :]
    ratings[rated_games] = 0

    top_games = np.argpartition(-ratings, kth=5)[:5]
    sorted_top_games = list(top_games[np.argsort(ratings[top_games])[::-1]])
    print(ratings[sorted_top_games])
    users_top_games.append([game for game in sorted_top_games])
    print(users[users["User index"] == user]["Username"].item(), sorted_top_games)
    games_ids = games_table[games_table.index.isin(sorted_top_games)].values

    print(jeux_clean[jeux_clean["Game id"].isin(games_ids)]["Game name year"])

    pred[user, rated_games] = true_ratings[user, rated_games].toarray()
    users_top_ratings.append([rating for rating in ratings])

#type(users_top_games[0])

In [ ]:
pred = pred[users_indices, :]

In [ ]:
indices = [2247, 1713, 1703, 1867, 30]
#[games_info["game index"].isin(indices)]
pred[0, indices]

In [ ]:
pred = U @ G.T - 2
pred = np.clip(pred, a_min=0, a_max=10) / (10 - 2)
pred = pred[users_indices]
pred[0, indices] * 8 + 2

In [ ]:
pred = np.clip(pred[users_indices], a_min=0, a_max=10) / 10
np.save("nnmf_prediction.npy", pred)

#pd.DataFrame({"Ratings":pred[users_indices].tolist(), "User index":users_indices})
#np.save("nnmf_prediction.npy", pred)
#print(pred)

#np.save("nnmf_prediction.npy", pred)

In [ ]:
users_indices

In [ ]:
mapping = {user_index : i for i, user_index in enumerate(users_indices)}
users.loc[:, "User index"] = users["User index"].map(mapping)

In [ ]:
users["Top games"] = users_top_games
users["Username"] = users.apply(lambda row : f"{row['Username']} ({row['Number reviews']} avis)", axis=1)
users = users.sort_values("Number reviews", ascending=False).drop(columns="Number reviews")
users

In [ ]:
users.to_json("users_info.json", orient="records", force_ascii=False)

In [ ]:
users

***
### For optimization

In [ ]:
# df_all = pd.read_json("games_info.json", orient="records")
# games_info = pd.read_csv("games_info.csv", index_col=0).drop(columns="Description")
# games_info = games_info.rename(columns={"Game id":"game id", "Game index":"game index", "Game name year":"game name year",
#                             "Type":"type", "Players":"players", "Age":"age", "Rating":"rating"}).merge(df_all, on="game index")
# games_info = games_info.drop(columns=["game_id"])
# games_info.loc[:, "theme"] = games_info["name"]
# games_info.to_json("games_info.json", orient="records", force_ascii=False)